## XBRL US API - Taxonomy relationship details  

### Authenticate for access token 
Click in the gray code cell below, then click the Run button above to execute the cell. Type your XBRL US Web account email, account password, Client ID, and secret as noted, pressing the Enter key on the keyboard after each entry.

XBRL US limits records returned for a query to improve efficiency; this script loops to collect all data from the Public Filings Database for a query. **Non-members might not be able to return all data for a query** - join XBRL US for comprehensive access - https://xbrl.us/join.

In [ ]:
print('Enter your XBRL US Web account email: ')
import os, re, sys, json
import requests
import pandas as pd
from IPython.display import display, HTML
import numpy as np
import getpass
from datetime import datetime
import urllib
from urllib.parse import urlencode
email = input()
password = getpass.getpass(prompt='Password: ')
clientid = getpass.getpass(prompt='Client ID: ')
secret = getpass.getpass(prompt='Secret: ')

body_auth = {'username' : ''.join(email), 
            'client_id': ''.join(clientid), 
            'client_secret' : ''.join(secret), 
            'password' : ''.join(password), 
            'grant_type' : 'password', 
            'platform' : 'ipynb' }

payload = urlencode(body_auth)
url = 'https://api.xbrl.us/oauth2/token'
headers = {"Content-Type": "application/x-www-form-urlencoded"}

res = requests.request("POST", url, data=payload, headers=headers)
auth_json = res.json()

if 'error' in auth_json:
    print ("\n\nThere was a problem generating an access token with these credentials. Run the first cell again to enter credentials.")
else:
    print ("\n\nYour access token expires in 60 minutes. After it expires, run the cell immediately below this one to generate a new token and continue to use the query cell. \n\nFor now, skip ahead to the section 'Make a Query'.")
access_token = auth_json['access_token']
refresh_token = auth_json['refresh_token']
newaccess = ''
newrefresh = ''
#print('access token: ' + access_token + ' refresh token: ' + refresh_token)

#### Refresh token 
The cell below is only needed to refresh an expired access token after 60 minutes. When the access token no longer returns results, run the cell below to refresh the access token or re-enter credentials by running the cell above. Until the refresh token process is needed, **skip ahead to _Make a Query_**. 


In [ ]:
token = token if newrefresh != '' else refresh_token 

refresh_auth = {'client_id': ''.join(clientid), 
            'client_secret' : ''.join(secret), 
            'grant_type' : 'refresh_token', 
            'platform' : 'ipynb', 
            'refresh_token' : ''.join(token) }
refreshres = requests.post(url, data=refresh_auth)
refresh_json = refreshres.json()
access_token = refresh_json['access_token']
refresh_token = refresh_json['refresh_token']#print('access token: ' + access_token + 'refresh token: ' + refresh_token)
print('Your access token is refreshed for 60 minutes. If it expires again, run this cell to generate a new token and continue to use the query cells below.')
print(access_token)

### Make a query
After the access token confirmation appears above, you can modify the query below and use the **_Cell >> Run_** menu option with the cell **immediately below this text** to run the query for updated results.

The data frame below contains information about the presentation linkbase of the 2023 US GAAP taxonomy.
  
Refer to XBRL API documentation at https://xbrlus.github.io/xbrl-api/#/Facts/getFactDetails for other endpoints and parameters to filter and return.

In [ ]:
# These variables establish starting points for the query built and executed below
offset_value = 0
res_df = []
endpoint = 'relationship' # this endpoint shows the relationship among elements in a linkbase

# Define the parameters of the query

# Taxonomy
# To get a list of available taxonomies for a specific year, paste URL below into a browser (eg. 2023 taxonomies)
# https://api.xbrl.us/api/v1/dts/search?fields=dts.id,dts.taxonomy-name.sort(DESC)&dts.version=2023
dts_id = [
		'699456',# 2023 US GAAP
		]

# Linkbase
# This parameter can be used to return details for the taxonomy calculation or presentation linkbases; use calculationLink, presentationLink, 
linkbase = [
		'presentationLink',
		]

# Report Section - this filter can be applied to limit results to a single specified report section
# Comment out the report title to run this query for all relationship results for the taxonomy
# To get a list of available network.role-descriptions for the 2023 US GAAP taxonomy, paste the URL below into a browser
# https://api.xbrl.us/api/v1/dts/699456/network/search?fields=network.role-description.sort(ASC)&unique
role_description = [
		'104000 - Statement - Statement of Financial Position, Classified',
		]

# Fields to return (multi-sort based on order)
fields = [
		# this is the list of the characteristics of the data being returned by the query; sort is applied by the order in which columns are returned
		'network.role-description.sort(ASC)',
		'relationship.tree-sequence.sort(ASC)',
		'relationship.tree-depth.sort(ASC)',
		'relationship.order.sort(ASC)',
		'relationship.source-name',
		'relationship.target-name',
		'relationship.target-concept-id',
		'relationship.target-is-abstract',
		'relationship.target-datatype',
		'relationship.weight',
		'relationship.target-label'
		]

params = {
		# this is the list of what's being queried against the endpoint
		'dts.id': ','.join(dts_id),
		'network.link-name': ','.join(linkbase),
		'network.role-description': ','.join(role_description),
		#the filter below removes abstract rows from the results; uncomment the next line to include this filter
		#'relationship.target-is-abstract': 'false',
		'fields': ','.join(fields)
		}

# Create query and loop for all results - code below does not need to be changed
search_endpoint = 'https://api.xbrl.us/api/v1/' + endpoint + '/search'
orig_fields = params['fields']

count = 0
query_start = datetime.now()
printed = False
while True:
    if not printed:
        printed = True
    res = requests.get(search_endpoint, params=params, headers={'Authorization' : 'Bearer {}'.format(access_token)})
    res_json = res.json()
    if 'error' in res_json:
        print('There was an error: {}'.format(res_json['error_description']))
        break

    print("up to", str(offset_value + res_json['paging']['limit']), "records are found so far ...")

    res_df += res_json['data']

    if res_json['paging']['count'] < res_json['paging']['limit']:
        print(" - this set contained fewer than the", res_json['paging']['limit'], "possible, only", str(res_json['paging']['count']), "records.")
        break
    else:
        offset_value += res_json['paging']['limit']
        if 100 == res_json['paging']['limit']:
                params['fields'] = orig_fields + ',' + endpoint + '.offset({})'.format(offset_value)
                if offset_value == 10 * res_json['paging']['limit']:
                        break
        elif 500 == res_json['paging']['limit']:
                params['fields'] = orig_fields + ',' + endpoint + '.offset({})'.format(offset_value)
                if offset_value == 4 * res_json['paging']['limit']:
                        break
        params['fields'] = orig_fields + ',' + endpoint + '.offset({})'.format(offset_value)

if not 'error' in res_json:
    current_datetime = datetime.now().replace(microsecond=0)
    time_taken = current_datetime - query_start
    index = pd.DataFrame(res_df).index
    total_rows = len(index)
    your_limit = res_json['paging']['limit']
    limit_message = "If the results below match the limit noted above, you might not be seeing all rows, and should consider upgrading (https://xbrl.us/access-token).\n"

    if your_limit == 100:
        print("\nThis non-Member account has a limit of " , 10 * your_limit, " rows per query from our Public Filings Database. " + limit_message)
    elif your_limit == 500:
        print("\nThis Basic Individual Member account has a limit of ", 4 * your_limit, " rows per query from our Public Filings Database. " + limit_message)

    print("\nAt " + current_datetime.strftime("%c") +  ", the query finished with  ", str(total_rows), "  rows returned in " + str(time_taken) + " for \n" +  urllib.parse.unquote(res.url))


    df = pd.DataFrame(res_df)
    # the .to_html formatting declarations are for presentation only; the data frame is unaffected - all rows and defined fields are available (use save to .csv)
    display(HTML(df.to_html(
		max_rows=10,
		columns=['network.role-description','relationship.tree-sequence','relationship.tree-depth','relationship.order','relationship.source-name','relationship.target-name','relationship.weight','relationship.target-label']
		).replace('<td>', '<td style="vertical-align:text-top;text-align:left;max-width:200px;word-wrap:break-word;">')
		))

up to 5000 records are found so far ...
 - this set contained fewer than the 5000 possible, only 689 records.

At Wed Feb  7 09:15:37 2024, the query finished with   689   rows returned in 0:00:02.441532 for 
https://api.xbrl.us/api/v1/relationship/search?dts.id=699456&network.link-name=presentationLink&network.role-description=104000+-+Statement+-+Statement+of+Financial+Position,+Classified&fields=network.role-description.sort(ASC),relationship.tree-sequence.sort(ASC),relationship.tree-depth.sort(ASC),relationship.order.sort(ASC),relationship.source-name,relationship.target-name,relationship.target-concept-id,relationship.target-is-abstract,relationship.target-datatype,relationship.weight,relationship.target-label


,network.role-description,relationship.tree-sequence,relationship.tree-depth,relationship.order,relationship.source-name,relationship.target-name,relationship.weight,relationship.target-label
0,"104000 - Statement - Statement of Financial Position, Classified",1,1,10,StatementOfFinancialPositionAbstract,StatementTable,None,Statement [Table]
1,"104000 - Statement - Statement of Financial Position, Classified",2,2,10,StatementTable,RestatementAxis,None,Revision of Prior Period [Axis]
2,"104000 - Statement - Statement of Financial Position, Classified",3,3,10,RestatementAxis,RestatementDomain,None,Revision of Prior Period [Domain]
3,"104000 - Statement - Statement of Financial Position, Classified",4,4,10,RestatementDomain,ScenarioPreviouslyReportedMember,None,Previously Reported [Member]
4,"104000 - Statement - Statement of Financial Position, Classified",5,4,20,RestatementDomain,RestatementAdjustmentMember,None,"Revision of Prior Period, Adjustment [Member]"
...,...,...,...,...,...,...,...,...
684,"104000 - Statement - Statement of Financial Position, Classified",685,7,40,MembersEquityAbstract,NotesReceivableByOwnerToLimitedLiabilityCompanyLLC,None,Notes Receivable by Owner to Limited Liability Company (LLC)
685,"104000 - Statement - Statement of Financial Position, Classified",686,7,50,MembersEquityAbstract,MembersEquity,None,"Members' Equity, Total"
686,"104000 - Statement - Statement of Financial Position, Classified",687,5,20,LimitedLiabilityCompanyLLCMembersEquityIncludingPortionAttributableToNoncontrollingInterestAbstract,MembersEquityAttributableToNoncontrollingInterest,None,Members' Equity Attributable to Noncontrolling Interest
687,"104000 - Statement - Statement of Financial Position, Classified",688,5,30,LimitedLiabilityCompanyLLCMembersEquityIncludingPortionAttributableToNoncontrollingInterestAbstract,LimitedLiabilityCompanyLlcMembersEquityIncludingPortionAttributableToNoncontrollingInterest,None,"Limited Liability Company (LLC) Members' Equity, Including Portion Attributable to Noncontrolling Interest, Total"


In [ ]:
# If you run this program locally, you can save the output to a file on your computer (modify D:\results.csv to your system)
df.to_csv(r"D:\results.csv",sep=",")

In [ ]:
# This routine iterates on each row of the dataframe to return additional concept details and saves the iterated rows to a data.json file in the same location as this notebook.
# This sample output is posted at https://raw.githubusercontent.com/xbrlus/xbrl-api-ipynb/python/data.json
# Comment out limit to return all rows in the dataframe 
# WARNING - using this routine for all taxonomy elements is not the best approach 
# For all elements, use a query like https://api.xbrl.us/api/v1/concept/search?dts.id=699456&fields=reference.role-definition,concept.id,concept.local-name,dts.id,parts.*

limit = 25
all_concepts = []
for index, row in df[:limit].iterrows():

    # Concept from dataframe
        target_id = str(row['relationship.target-concept-id'])

    # Fields to return (multi-sort based on order)
        fields = [
            # this is the list of the characteristics of the data being returned by the query
            'label.text',
            'label.role-short',
            'concept.id',
            'concept.local-name',
            'parts.order.sort(ASC)',
            'parts.*'
            ]

        params = {
            'fields': ','.join(fields)
            }

        # Create query and loop for all results - code below does not need to be changed
        search_endpoint = 'https://api.xbrl.us/api/v1/concept/search?dts.id=699456&concept.id=' + target_id
        orig_fields = params['fields']
        concept = requests.get(search_endpoint, params=params, headers={'Authorization' : 'Bearer {}'.format(access_token)})
        concept_json = concept.json()
        print(concept_json['data'])
        all_concepts += concept_json['data'] 
        with open('data.json', 'w') as f:
                json.dump(all_concepts, f, indent=4)

[{'concept.id': 39601986, 'concept.local-name': 'StatementTable', 'reference': {'data': [{'parts': {'data': [{'parts.order': 1, 'parts.namespace': 'http://fasb.org/tin-part/2023', 'parts.local-name': 'elementCreationTaxonomyVersion', 'parts.part-value': '2008'}], 'paging': {'limit': 5000, 'offset': 0}}, 'dts.id': 699456, 'reference.id': 337847568}, {'parts': {'data': [{'parts.order': 1, 'parts.namespace': 'http://fasb.org/cn-part/2023', 'parts.local-name': 'TaxonomyVersion', 'parts.part-value': '2023'}, {'parts.order': 2, 'parts.namespace': 'http://fasb.org/cn-part/2023', 'parts.local-name': 'SourceName', 'parts.part-value': 'BDC:Taxonomy Technical Improvement'}, {'parts.order': 3, 'parts.namespace': 'http://fasb.org/cn-part/2023', 'parts.local-name': 'ModifiedReferences', 'parts.part-value': 'true'}], 'paging': {'limit': 5000, 'offset': 0}}, 'dts.id': 699456, 'reference.id': 337847673}, {'parts': {'data': [{'parts.order': 1, 'parts.namespace': 'http://fasb.org/codification-part/2023',